# Adding Short-Term Memory to the Agent

In this notebook, we will learn how to add memory to our SQL AI agent so users can ask natural follow-up questions without restating all the context.


<figure>
 <img src="../assets/chapter_2.png" width="60%" align="center"/></a>
<figcaption> Prompt Template Architecture </figcaption>
</figure>

<br>
<br />

## Setting the Database Connection

The below code enables us to connect to Postgres (or DuckDB) using the `get_ibis_connection` function:

In [1]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection


Setting a connection to the Postgres database:

In [2]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

Or, setting a connection to the DuckDB database:

In [3]:
# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

Next, we will extract the table attributes using the `get_tbl_attr` function:

In [4]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema


## Setting the LLM Client

We will use the `ChatOpenAI` to set the LLM client connection:

In [5]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(base_url=base_url, api_key=api_key, temperature=0, model=model)


Extracting distinct values from the categorical fields:

In [6]:
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt

distinct_values = get_character_distinct_values(
    con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
)
distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)


## Add Memory to the Prompt Template

We will update the messages object by adding a new component - message place holder using the `MessagesPlaceholder` method: 

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [
    ("system", system_template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", user_template),
]

prompt_template = ChatPromptTemplate.from_messages(messages)


Setting up the memory object:

In [8]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()


Create the chain:

In [9]:
chain = prompt_template | llm


Last but non least, updating the `basic_sql_agent` function - adding the `chat_history` as input argument:

In [10]:
def basic_sql_agent(chain, question, tbl_name, schema, con, chat_history, additional_context=""):
    
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
            "chat_history": chat_history.messages,
        }
    )
    query = llm_output.content

    chat_history.add_user_message(question)
    chat_history.add_ai_message(query)
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_" * 60)
    output = con.sql(query).execute()
    print(output)
    return output, chat_history


In [11]:
print(chat_history)

Let's test it!

In [12]:
output, chat_history = basic_sql_agent(
    chain,
    question="How many passengers departed during 2024 via terminal 1?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    chat_history=chat_history,
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned'
  AND "Terminal" = 'Terminal 1'
  AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________
  Total Passengers
0          7053492


In [13]:
print(chat_history)

Human: How many passengers departed during 2024 via terminal 1?
AI: SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned'
  AND "Terminal" = 'Terminal 1'
  AND EXTRACT(YEAR FROM "Date") = 2024;


In [14]:
output, chat_history = basic_sql_agent(
    chain,
    question="And via terminal 2?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    chat_history=chat_history,
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned'
  AND "Terminal" = 'Terminal 2'
  AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________
  Total Passengers
0          3086869


In [15]:
print(chat_history)

Human: How many passengers departed during 2024 via terminal 1?
AI: SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned'
  AND "Terminal" = 'Terminal 1'
  AND EXTRACT(YEAR FROM "Date") = 2024;
Human: And via terminal 2?
AI: SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned'
  AND "Terminal" = 'Terminal 2'
  AND EXTRACT(YEAR FROM "Date") = 2024;
